In [2]:
import pandas as pd
from pathlib import Path

# =========================
# ΡΥΘΜΙΣΕΙΣ
# =========================

INPUT_DIR = Path("E:/IDEAL/processed/hourly_per_home")
OUTPUT_DIR = Path("E:/IDEAL/processed/hourly_per_home_filled")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD_NAN = 100 # όριο NaN για επιλογή μεθόδου


# =========================
# ΒΟΗΘΗΤΙΚΗ ΣΥΝΑΡΤΗΣΗ
# =========================

def fill_from_next_year(df, ts_col="timestamp", col="internal_temperature"):
    """
    Συμπληρώνει NaN τιμές από το ίδιο timestamp του επόμενου έτους (t + 1 year), αν υπάρχει.
    """
    df = df.copy()

    value_map = (
        df.dropna(subset=[col])
          .set_index(ts_col)[col]
          .to_dict()
    )

    filled_count = 0

    for idx, row in df[df[col].isna()].iterrows():
        t_next = row[ts_col] + pd.DateOffset(years=1)
        if t_next in value_map:
            df.at[idx, col] = value_map[t_next]
            filled_count += 1

    return df, filled_count


# =========================
# ΚΥΡΙΑ ΕΠΕΞΕΡΓΑΣΙΑ
# =========================

report_rows = []

for csv_file in sorted(INPUT_DIR.glob("home*_hourly.csv")):
    home_id = csv_file.stem.replace("_hourly", "")
    df = pd.read_csv(csv_file, parse_dates=["timestamp"])

    total_rows = len(df)
    nan_before = df["internal_temperature"].isna().sum()
    seasonal_filled = 0

    # Επιλογή μεθόδου
    if nan_before > THRESHOLD_NAN:
        # 1) Seasonal fill από επόμενο έτος
        df, seasonal_filled = fill_from_next_year(
            df, ts_col="timestamp", col="internal_temperature"
        )

        # 2) Interpolation για ό,τι έμεινε
        df = df.sort_values("timestamp")
        s = df.set_index("timestamp")["internal_temperature"]
        s = s.interpolate(method="time", limit_direction="both")
        df["internal_temperature"] = s.reset_index(drop=True)

        method_used = "seasonal_next_year + interpolate(time)"
    else:
        # Μόνο interpolation
        df = df.sort_values("timestamp")
        s = df.set_index("timestamp")["internal_temperature"]
        s = s.interpolate(method="time", limit_direction="both")
        df["internal_temperature"] = s.reset_index(drop=True)

        method_used = "interpolate(time)"

    # Στρογγυλοποίηση σε 1 δεκαδικό
    df["internal_temperature"] = df["internal_temperature"].round(1)

    nan_after = df["internal_temperature"].isna().sum()

    # Αποθήκευση
    out_path = OUTPUT_DIR / csv_file.name
    df.to_csv(out_path, index=False)

    # Αναφορά
    report_rows.append({
        "home_id": home_id,
        "rows": total_rows,
        "nan_internal_before": nan_before,
        "seasonal_filled_count": seasonal_filled,
        "nan_internal_after": nan_after,
        "method": method_used
    })

# =========================
# ΤΕΛΙΚΟ REPORT
# =========================

report = (
    pd.DataFrame(report_rows)
      .sort_values("nan_internal_before", ascending=False)
)

report

,home_id,rows,nan_internal_before,seasonal_filled_count,nan_internal_after,method
57,home167,10576,5821,3732,0,seasonal_next_year + interpolate(time)
173,home290,3265,84,0,0,interpolate(time)
220,home64,13452,20,0,0,interpolate(time)
136,home253,4106,7,0,0,interpolate(time)
131,home248,4919,7,0,0,interpolate(time)
...,...,...,...,...,...,...
242,home86,11666,0,0,0,interpolate(time)
246,home91,5481,0,0,0,interpolate(time)
250,home96,8468,0,0,0,interpolate(time)
252,home98,11290,0,0,0,interpolate(time)
